Ответ: 11

# 1. Загрузите картинку parrots.jpg. Преобразуйте изображение, приведя все значения в интервал от 0 до 1. Для этого можно воспользоваться функцией img_as_float из модуля skimage. Обратите внимание на этот шаг, так как при работе с исходным изображением вы получите некорректный результат.

In [1]:
import pandas as pd
import numpy as np
from skimage.io import imread
from skimage import img_as_float
from sklearn.cluster import KMeans
import math
import pylab

image = imread("/content/parrots.jpg")
image_float = img_as_float(image)

# 2. Создайте матрицу объекты-признаки: характеризуйте каждый пиксель тремя координатами - значениями интенсивности в пространстве RGB.

In [3]:
w, h, d = image_float.shape
X = image_float.reshape((w * h, d))

# 3. Запустите алгоритм K-Means с параметрами init=’k-means++’ и random_state=241. После выделения кластеров все пиксели, отнесенные в один кластер, попробуйте заполнить двумя способами: медианным и средним цветом по кластеру.

In [4]:
n_clusters = 10
kmeans = KMeans(init='k-means++', random_state=241, n_clusters=n_clusters)
kmeans.fit(X)

labels = kmeans.labels_
X_mean = np.copy(X)
X_median = np.copy(X)

for cluster in range(n_clusters):
    cluster_pixels = X[labels == cluster]
    X_mean[labels == cluster] = np.mean(cluster_pixels, axis=0)
    X_median[labels == cluster] = np.median(cluster_pixels, axis=0)

# 4. Измерьте качество получившейся сегментации с помощью метрики PSNR. Эту метрику нужно реализовать самостоятельно (см. определение).

In [5]:
def calculate_psnr(image_original, image_compressed):
    mse = np.mean((image_original - image_compressed) ** 2)
    if mse == 0:
        return float('inf')
    # Формула PSNR: 20 * log10(MAX) - 10 * log10(MSE), где MAX = 1.0
    psnr = 10 * math.log10(1.0 / mse)
    return psnr

img_mean = X_mean.reshape((w, h, d))
img_median = X_median.reshape((w, h, d))

psnr_mean = calculate_psnr(image_float, img_mean)
psnr_median = calculate_psnr(image_float, img_median)

print(f"PSNR (Mean): {psnr_mean:.2f}")
print(f"PSNR (Median): {psnr_median:.2f}")

PSNR (Mean): 19.54
PSNR (Median): 19.23


# 5. Найдите минимальное количество кластеров, при котором значение PSNR выше 20 (можно рассмотреть не более 20 кластеров, но не забудьте рассмотреть оба способа заполнения пикселей одного кластера). Это число и будет ответом в данной задаче.


In [8]:
for n in range(1, 21):
    kmeans = KMeans(init='k-means++', random_state=241, n_clusters=n)
    kmeans.fit(X)

    X_mean = np.copy(X)
    X_median = np.copy(X)

    for cluster in range(n):
        cluster_pixels = X[kmeans.labels_ == cluster]
        X_mean[kmeans.labels_ == cluster] = np.mean(cluster_pixels, axis=0)
        X_median[kmeans.labels_ == cluster] = np.median(cluster_pixels, axis=0)

    psnr_mean = calculate_psnr(image_float, X_mean.reshape((w, h, d)))
    psnr_median = calculate_psnr(image_float, X_median.reshape((w, h, d)))

    if psnr_mean > 20 or psnr_median > 20:
        print(n)
        break

11
